In [14]:
from Repositories.Members.IReadableMemberList import IReadableMemberList
import __init__
from abc import ABCMeta, abstractmethod
from typing import Optional, Tuple, List
from result import Result, Err, Ok

from Domains.Members import *
from Repositories.Members import *
from Builders.Members import *
from uuid import UUID

import pymysql

from icecream import ic


class MySqlGetPrivacy(IReadableMember):
    def __init__(self, name_padding: str = "log_"):
        self.name_padding = name_padding

    def connect(self):
        from get_config_data import get_mysql_dict

        sql_config = get_mysql_dict()
        return pymysql.connect(
            host=sql_config["host"],
            user=sql_config["user"],
            password=sql_config["password"],
            db=sql_config["database"],
            charset=sql_config["charset"],
        )

    def get_padding_name(self, name: str) -> str:
        return f"{self.name_padding}{name}"

    def get_privacy(self, member_id: MemberID) -> Result[Privacy, str]:
         
        connection = self.connect()
        user_table_name = self.get_padding_name("user")
        
        try:
        
            with connection.cursor() as cursor:
                
                query = f"""
                    SELECT id, name, phone, email, address, pay_account, company_registration_number
                    FROM {user_table_name}
                    WHERE id = %s
                    ORDER BY seq DESC
                    LIMIT 1
                """           
                cursor.execute(query, (member_id.get_id(), ))
                           
                result = cursor.fetchone()
                
                if not result:
                    return Err("Member not found")

                id, name, phone, email, address, pay_account, company_registration_number = result
                
                privacy = Privacy(
                    id=MemberIDBuilder().set_uuid(id).build(),
                    name=name,  
                    phone=phone,
                    email=email,
                    address=address,
                    pay_account=pay_account,
                    company_registration_number=company_registration_number
                )
                

                connection.commit()

                return Ok(privacy)
            
        except Exception as e:
            print(e)
            connection.rollback()
            return Err(str(e))

        finally:
            connection.close()


my_sql_get_privacy = MySqlGetPrivacy()

member_id = MemberIDBuilder().set_uuid("2a24d8196dad43eebfb5b1051bc531c2").build()

result = my_sql_get_privacy.get_privacy(member_id)

if result.is_ok():
    privacy = result.unwrap()
    print("Privacy information:", privacy)
else:
    print("Error occurred:", result.unwrap_err())

Privacy information: Privacy(id=MemberID(uuid=UUID('2a24d819-6dad-43ee-bfb5-b1051bc531c2'), sequence=-1), name='장예서', phone='01067541234', email='bstax@daum.com', address='전라북도 익산', pay_account=None, company_registration_number=None)


In [15]:
# MySqlGetPrivacy 클래스의 인스턴스 생성
my_sql_get_privacy = MySqlGetPrivacy()

# 주어진 문자열로 MemberID 인스턴스 생성
member_id = MemberID("0fb37d1cbb1e43c4a6aed9438c835f52")

# get_privacy 메서드 호출
result = my_sql_get_privacy.get_privacy(member_id)

if result.is_ok():
    privacy = result.unwrap()
    print("Privacy information:", privacy)
else:
    print("Error occurred:", result.unwrap_err())


'str' object has no attribute 'hex'
Error occurred: 'str' object has no attribute 'hex'
